# Class 3/09/26 Notes #

## A Simple Ollama Example ##

The [Ollama documentation](https://docs.ollama.com/capabilities/tool-calling#python) provides a *very simple* example of tool calling with Ollama.

### Idea: ###

+ **Request Format:** You define tools in JSON and include them in the tools parameter of your ollama.chat call.
+ **Response Structure:** When a tool is needed, the assistant's message will contain a tool_calls object with the function name and arguments, instead of text in the content field.

### Workflow: ###

1. Send user message + tool definitions to Ollama.
2. Model returns tool_calls.
3. Your application runs the tool (function).
4. Send the tool output back to Ollama with the role tool.
5. Model generates the final response.


In [14]:
from ollama import chat

def get_temperature(city: str) -> str:
  """Get the current temperature for a city
  
  Args:
    city: The name of the city

  Returns:
    The current temperature for the city
  """
  temperatures = {
    "New York": "32°C",
    "London": "15°C",
    "Tokyo": "18°C",
  }
  return temperatures.get(city, "Unknown")

messages = [{"role": "user", "content": "What is the temperature in New York?"}]

# pass functions directly as tools in the tools list or as a JSON schema
response = chat(model="qwen3", messages=messages, tools=[get_temperature], think=True)

messages.append(response.message)
if response.message.tool_calls:
  # only recommended for models which only return a single tool call
  call = response.message.tool_calls[0]
  result = get_temperature(**call.function.arguments)
  # add the tool result to the messages
  messages.append({"role": "tool", "tool_name": call.function.name, "content": str(result)})

  final_response = chat(model="qwen3", messages=messages, tools=[get_temperature], think=True)
  print(final_response.message.content)

The current temperature in New York is **32°C**. Let me know if you need more weather details! 🌡️


## A Slightly Less Simple Ollama Example ##

See [this medium article](https://medium.com/@danushidk507/ollama-tool-calling-8e399b2a17a8) and [this medium article](https://medium.com/@meirgotroot/ollama-support-for-tool-calling-186bcfeb892f) for more detailed examples. 

The idea for the first example (stock calling):

1. User Query → “What is the current stock price of Apple?”
2. AI Response:

+ Detects it needs a stock price → Calls get_stock_price_tool.
+ Retrieves stock data (235.06).
+ Integrates the result into the conversation.

In [15]:
import ollama
import yfinance as yf
from typing import Dict, Any, Callable

In [16]:
def get_stock_price(symbol: str) -> float:
    ticker = yf.Ticker(symbol)
    price_attrs = ['regularMarketPrice', 'currentPrice', 'price']
    
    for attr in price_attrs:
        if attr in ticker.info and ticker.info[attr] is not None:
            return ticker.info[attr]
            
    fast_info = ticker.fast_info
    if hasattr(fast_info, 'last_price') and fast_info.last_price is not None:
        return fast_info.last_price
        
    raise Exception("Could not find valid price data")

In [17]:
get_stock_price_tool = {
    'type': 'function',
    'function': {
        'name': 'get_stock_price',
        'description': 'Get the current stock price for any symbol',
        'parameters': {
            'type': 'object',
            'required': ['symbol'],
            'properties': {
                'symbol': {'type': 'string', 'description': 'The stock symbol (e.g., AAPL, GOOGL)'},
            },
        },
    },
}

In [18]:
prompt = 'What is the current stock price of Apple?'
print('Prompt:', prompt)

Prompt: What is the current stock price of Apple?


In [19]:
available_functions: Dict[str, Callable] = {
    'get_stock_price': get_stock_price,
}

In [20]:
response = ollama.chat(
    'llama3.1',
    messages=[{'role': 'user', 'content': prompt}],
    tools=[get_stock_price_tool],
)

In [21]:
print(response.message.content)

In [22]:
if response.message.tool_calls:
    for tool in response.message.tool_calls:
        if function_to_call := available_functions.get(tool.function.name):
            print('Calling function:', tool.function.name)
            print('Arguments:', tool.function.arguments)
            print('Function output:', function_to_call(**tool.function.arguments))
        else:
            print('Function', tool.function.name, 'not found')

Calling function: get_stock_price
Arguments: {'symbol': 'AAPL'}
Function output: 255.14


## A Simple Langchain Tool Calling Example ##

(Pretty much the same as above.)

In [42]:
import json
from langchain_ollama import ChatOllama
from langchain.tools import tool
from langchain.messages import HumanMessage, ToolMessage
# 1. Define a tool using the @tool decorator
@tool
def get_current_weather(location: str, unit: str = "fahrenheit") -> str:
    """
    Get the current weather in a given location.

    Args:
        location: The city and state, e.g. San Francisco, CA
        unit: The unit of temperature, e.g. celsius or fahrenheit
    """
    # Example static data, replace with real API call in production
    weather_data = {
        "San Francisco, CA": f"70°{unit}",
        "New York, NY": f"65°{unit}",
        "London": f"15°{unit}",
    }
    return weather_data.get(location, "Unknown weather")

# 2. Instantiate a ChatOllama model and bind the tool
# Make sure to use a model that supports tool calling, like llama3.1 or qwen
llm = ChatOllama(model="llama3.1")
llm_with_tools = llm.bind_tools([get_current_weather])

# 3. Define the conversation loop
def run_conversation(prompt):
    messages = [HumanMessage(content=prompt)]
    ai_msg = llm_with_tools.invoke(messages)
    messages.append(ai_msg)

    # Check if the model decided to call a tool
    if ai_msg.tool_calls:
        print(f"AI decided to call tool(s): {ai_msg.tool_calls}")
        # Execute the tool calls
        for tool_call in ai_msg.tool_calls:
            # Run the actual Python function
            observation = get_current_weather.invoke(tool_call)
            print(f"Tool observation: {observation}")
            # Add the tool result back to the messages as a ToolMessage
            messages.append(ToolMessage(content=observation, tool_call_id=tool_call['id']))
        
        # Invoke the model again with the new context (tool results)
        final_response = llm_with_tools.invoke(messages)
        return final_response.content
    else:
        return ai_msg.content

# 4. Run the example
user_prompt = "What's the weather in San Francisco, CA and London?"
response = run_conversation(user_prompt)
print(f"\nFinal AI response: {response}")


AI decided to call tool(s): [{'name': 'get_current_weather', 'args': {'location': 'San Francisco, CA', 'unit': 'celsius'}, 'id': 'c2f10be5-fcd4-454a-9098-833ec8f8e907', 'type': 'tool_call'}, {'name': 'get_current_weather', 'args': {'location': 'London', 'unit': 'celsius'}, 'id': '769bf8c8-7ea9-4b38-879e-900545e1483d', 'type': 'tool_call'}]
Tool observation: content='70°celsius' name='get_current_weather' tool_call_id='c2f10be5-fcd4-454a-9098-833ec8f8e907'
Tool observation: content='15°celsius' name='get_current_weather' tool_call_id='769bf8c8-7ea9-4b38-879e-900545e1483d'

Final AI response: The current weather in San Francisco, CA is 70°Celsius, and the current weather in London is 15°Celsius.


### Another Example  ###

Same idea, slightly different code.   See [this page](https://docs.langchain.com/oss/python/langchain/models#tool-execution-loop) and [this page](https://docs.langchain.com/oss/python/langchain/tools) on the langchain docs site for the general work flow. 


In [41]:
from langchain_ollama import ChatOllama
from langchain.tools import tool
from langchain.messages import HumanMessage
import json

# 1. Define a tool using the @tool decorator
@tool
def get_current_weather(location: str, unit: str = "fahrenheit") -> str:
    """
    Get the current weather in a given location.
    
    Args:
        location (str): The city and state, e.g. "San Francisco, CA"
        unit (str): The unit of temperature, either "celsius" or "fahrenheit"
    """
    # In a real application, you would call an external API here.
    # For this example, we use hardcoded data.
    weather_data = {
        "New York, NY": "Sunny, 75°F",
        "London": "Cloudy, 18°C",
        "Tokyo, Japan": "Rainy, 22°C",
    }
    return weather_data.get(location, "Unknown weather")

# 2. Instantiate the ChatOllama model and bind the tool
# Ensure the model specified (e.g., "llama3.1") is running in your local Ollama instance
llm = ChatOllama(model="qwen3")
llm_with_tools = llm.bind_tools([get_current_weather]) # Binds the function as a tool

# 3. Invoke the model with a user prompt
prompt = "What's the weather like in New York and Tokyo?"
ai_msg = llm_with_tools.invoke(prompt)

print(f"AI Message: {ai_msg}\n")

# 4. Process the tool calls
if ai_msg.tool_calls:
    print("Tool calls detected:\n")
    for tool_call in ai_msg.tool_calls:
        print(f"  Calling tool: {tool_name} with args: {tool_args}")
        tool_result = get_current_weather.invoke(tool_call)
        print(f" Tool response: {tool_result}\n\n")


AI Message: content='' additional_kwargs={} response_metadata={'model': 'qwen3', 'created_at': '2026-03-09T15:42:13.723860885Z', 'done': True, 'done_reason': 'stop', 'total_duration': 13725952938, 'load_duration': 183260784, 'prompt_eval_count': 192, 'prompt_eval_duration': 43163409, 'eval_count': 1208, 'eval_duration': 11220892854, 'logprobs': None, 'model_name': 'qwen3', 'model_provider': 'ollama'} id='lc_run--019cd343-293d-7100-8776-19f129edfa2b-0' tool_calls=[{'name': 'get_current_weather', 'args': {'location': 'New York, NY', 'unit': 'fahrenheit'}, 'id': 'eb8e6eae-7844-4deb-89b4-5a1e0fd7e093', 'type': 'tool_call'}, {'name': 'get_current_weather', 'args': {'location': 'Tokyo, Japan', 'unit': 'celsius'}, 'id': 'a82c5d72-2fc0-4de1-b858-65a66d618163', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 192, 'output_tokens': 1208, 'total_tokens': 1400}

Tool calls detected:

  Calling tool: get_current_weather with args: {'location': 'Tokyo, Japan', 'unit': 'cel

## A "Conceptual" RAG Example ##

RAG usual Langchain is discussed in more detail [here](https://docs.langchain.com/oss/python/langchain/rag).  The idea here is to see how the RAG steps mentioned in lectures are implemented in LangChain.

In [52]:
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_classic.chains import create_retrieval_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

# 1. Load Documents
# Use a document loader (e.g., for a PDF file named 'manual.pdf')
# LangChain provides many [document loaders](https://api.python.langchain.com)
loader = PyPDFLoader("illinois-drivers-manual.pdf")
docs = loader.load()

# 2. Split Documents
# Use a text splitter to divide the documents into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(docs)

# 3. Create Embeddings and Vector Store
# Create embeddings for the chunks using an embedding model
embeddings = OllamaEmbeddings(model="llama3.1")
# Store the embeddings in a vector database (FAISS is used here as an example)
vectorstore = FAISS.from_documents(chunks, embeddings)

# 4. Define a Retriever
# Convert the vector store into a retriever
retriever = vectorstore.as_retriever()

# 5. Define the Prompt Template
# A prompt instructs the LLM to use the provided context for answering
prompt = ChatPromptTemplate.from_template("""
Answer the following question based only on the provided context:

<context>
{context}
</context>

Question: {input}
""")

# 6. Initialize LLM
llm = ChatOllama(model="llama3.1", temperature=0.1)

# 7. Create the RAG Chain
# Combine the components into a single chain
question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

# 8. Invoke the Chain with a Question
user_query = "How old do you have to be the drive in the state of Illinois?"
response = rag_chain.invoke({"input": user_query})
print(response["answer"])


According to the provided context, the minimum age to drive in the state of Illinois is:

* 16 years old for a Class 3 low-speed electric bicycle
* 16 years old for a scooter or moped (with certain requirements, such as having a seat and footrest for the passenger)
* 21 years old for operating a motor vehicle with a blood-alcohol concentration of .08% or more (resulting in a minimum 6-month suspension of driving privileges)

Note that there may be additional requirements or restrictions for certain types of vehicles or circumstances, but these are the minimum ages mentioned in the provided context.
